# Semantic similarity · review
Skor kontinu 0–5, lima pendekatan. Data [STSB terjemahan Indonesia](https://huggingface.co/datasets/LazarusNLP/stsb_mt_id): validation resmi dipakai sebagai pool pengembangan karena tidak ada train resmi; test resmi dipertahankan. Terjemahan mesin dapat mengubah makna. Pearson/Spearman utama, MAE/RMSE untuk skala; parafrase hanyalah turunan dengan batas konfigurasi.

## 00 · Konfigurasi

Ubah sumber/kolom/ukuran sampel di sini. Tidak ada grafik otomatis.

In [1]:
SEED,TOP=42,15
RESULT_NAME='hasil_similarity_review'
N_POOL,N_TEST,EDA_N=1500,1379,3000
EPOCHS,PAIR_MAX_LEN=3,256
PARAPHRASE_MIN=4.0

## 01 · Lingkungan & preprocessing teknis

Safe cleaning: encoding, NFC, HTML entity, invisible, whitespace. Pertahankan angka, negasi, case, entitas dan bahasa. Tanpa stemming/stopword removal pada encoder.

In [2]:
import os,sys,subprocess,time,json,re,html,unicodedata,random,gc
from pathlib import Path
from collections import Counter
assert Path('/kaggle/input').exists(),'Jalankan di Kaggle.'
os.environ.update(USE_TF='0',USE_FLAX='0',TOKENIZERS_PARALLELISM='false',HF_HUB_DISABLE_PROGRESS_BARS='1')
subprocess.run([sys.executable,'-m','pip','install','-q','ftfy>=6.3,<7','rank-bm25>=0.2,<1','sentencepiece>=0.2,<1','langid==1.1.6'],check=True)
import numpy as np,pandas as pd,torch,ftfy
from IPython.display import display
from scipy.special import softmax
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.metrics import f1_score,classification_report,confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import Ridge,LogisticRegression
from transformers import AutoTokenizer,AutoModel,AutoModelForSequenceClassification
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.set_num_threads(2)
assert torch.cuda.is_available(),'Aktifkan GPU Kaggle.'
OUT=Path('/kaggle/working')/RESULT_NAME;OUT.mkdir(exist_ok=True)
pd.set_option('display.max_columns',None,'display.max_colwidth',120)
START=time.perf_counter()

In [3]:
def show(title,table):
    print('\n'+title)
    display((table.to_frame() if isinstance(table,pd.Series) else table).round(4))

def distribution(values):
    n=values.value_counts(dropna=False)
    return pd.DataFrame({'n':n,'%':100*n/(len(values) or np.nan)})

def clean(value):
    if not isinstance(value,str): return ''
    repaired=ftfy.fix_text(value,unescape_html=False,uncurl_quotes=False,fix_latin_ligatures=False,
        fix_character_width=False,remove_control_chars=False,normalization=None)
    text=unicodedata.normalize('NFC',html.unescape(repaired))
    text=text.translate(dict.fromkeys(map(ord,'\ufeff\u200b\u2060\x00'),None))
    text=re.sub(r'[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]',' ',text)
    return re.sub(r'\s+',' ',text).strip()

def key(value): return clean(value).casefold()
def cap(frame,n): return frame.sample(min(n,len(frame)),random_state=SEED).reset_index(drop=True) if n else frame.reset_index(drop=True)
def locate(folder,repo):
    matches=list(Path('/kaggle/input').glob('**/models/'+folder+'/config.json'))
    return str(matches[0].parent) if matches else repo
NOISE={'url':r'https?://|www\.', 'mention':r'@\w+','hashtag':r'#\w+','angka':r'\d',
 'emoji/simbol':r'[\U0001F300-\U0001FAFF\u2600-\u27BF]','huruf_ulang':r'([^\W\d_])\1{2,}',
 'baca_ulang':r'[!?.,]{2,}','html':r'</?[a-zA-Z][^>]*>','encoding':r'\ufffd|Ã.|Â.',
 'non_ascii':r'[^\x00-\x7f]','spasi_ulang':r' {2,}','invisible':r'[\u200b\ufeff]',
 'KAPITAL':r'(?-i:\b[A-Z]{3,}\b)','huruf_angka':r'\b(?=\w*[a-z])(?=\w*\d)\w+\b',
 'slang_contoh':r'\b(?:gk|gak|nggak|ga|bgt|yg|utk|udh|klo|jg)\b',
 'negasi':r'\b(?:tidak|tak|bukan|belum|jangan|tanpa|gak|nggak|ga|not|never)\b'}

In [4]:
def text_eda(values,title):
    valid=values.where(values.map(lambda v:isinstance(v,str)),'').fillna('')
    show(title+' kualitas',pd.Series({'n':len(values),'null':values.isna().sum(),'kosong/spasi':valid.str.strip().eq('').sum(),
        'unik_terisi':valid[valid.str.strip().ne('')].nunique(),'duplikat_ekstra_terisi':valid[valid.str.strip().ne('')].duplicated().sum()}))
    show(title+' length',pd.DataFrame({'karakter':valid.str.len(),'kata':valid.str.split().str.len()}).describe(percentiles=[.01,.5,.9,.95,.99]).T)
    sample=valid[valid.str.strip().ne('')].sample(min(EDA_N,valid.str.strip().ne('').sum()),random_state=SEED)
    tokens=sample.str.casefold().str.findall(r'[^\W\d_]+')
    count=pd.Series(Counter(w for seq in tokens for w in seq),dtype='int64').sort_values(ascending=False)
    show(title+' vocabulary sampel',pd.Series({'n_sample':len(sample),'token':count.sum(),'unik':len(count),
        'hapax':count.eq(1).sum(),'hapax_%_tipe':100*count.eq(1).mean(),'hapax_%_token':100*count.eq(1).sum()/(count.sum() or np.nan)}))
    show(title+' top words',count.head(TOP).rename('n').to_frame().assign(pct=count.head(TOP)/(count.sum() or np.nan)*100))
    for size in [2,3]:
        grams=pd.Series(Counter(' '.join(seq[i:i+size]) for seq in tokens for i in range(len(seq)-size+1)),dtype='int64').sort_values(ascending=False)
        show(title+f' top {size}-gram',grams.head(TOP))
    flags=pd.DataFrame({name:valid.str.count(regex,flags=re.I).gt(0) for name,regex in NOISE.items()})
    show(title+' noise · % semua baris, dapat overlap',pd.DataFrame({'n':flags.sum(),'%':flags.mean()*100}))
    terms=count.index.to_series().astype('string')
    weird=terms.str.len().ge(20)|terms.str.count(r'([^\W\d_])\1{2,}').gt(0)
    show(title+' kandidat kata aneh, bukan pasti typo',count[weird].head(TOP))
    from langid.langid import LanguageIdentifier,model
    from threadpoolctl import threadpool_limits
    detector=LanguageIdentifier.from_modelstring(model,norm_probs=True)
    lang_sample=sample.sample(min(150,len(sample)),random_state=SEED)
    with threadpool_limits(limits=1):
        lang=lang_sample.map(lambda s:detector.classify(s[:3000]) if len(s.split())>=5 else ('tidak_pasti',np.nan))
    show(title+' estimasi bahasa · % sampel, bukan gold',distribution(lang.map(lambda p:p[0] if p[1]>=.8 else 'tidak_pasti')))

In [5]:
def prepare_pairs(raw,a,b,target):
    show('Missing sebelum drop',pd.DataFrame({'n':raw.isna().sum(),'%':raw.isna().mean()*100}))
    frame=raw.copy();frame['a']=frame[a].map(clean);frame['b']=frame[b].map(clean)
    frame['y']=pd.to_numeric(frame[target],errors='coerce')
    ok=frame.a.ne('')&frame.b.ne('')&frame.y.notna()&np.isfinite(frame.y)
    print('Dibuang invalid:',int((~ok).sum()))
    return frame.loc[ok].reset_index(drop=True)

def endpoints(frame): return set(frame.a.map(key))|set(frame.b.map(key))
def purge(frame,blocked):
    keep=~frame.a.map(key).isin(blocked)&~frame.b.map(key).isin(blocked)
    print('Pasangan dikeluarkan karena teks lintas split:',int((~keep).sum()))
    return frame.loc[keep].reset_index(drop=True)

def pair_groups(frame):
    parent={}
    def root(x):
        parent.setdefault(x,x)
        while parent[x]!=x:parent[x]=parent[parent[x]];x=parent[x]
        return x
    for a,b in zip(frame.a.map(key),frame.b.map(key)):parent[root(a)]=root(b)
    return pd.factorize([root(x) for x in frame.a.map(key)])[0]

def pair_features(frame):
    rows=[]
    for a,b in zip(frame.a,frame.b):
        x,y=set(a.casefold().split()),set(b.casefold().split());union=x|y
        rows.append([len(x&y)/max(1,len(union)),len(x-y)/max(1,len(x)),len(y-x)/max(1,len(y)),len(a)/max(1,len(b)),
            int(bool(re.search(NOISE['negasi'],a,re.I)))!=int(bool(re.search(NOISE['negasi'],b,re.I))),
            len(set(re.findall(r'\d+',a))^set(re.findall(r'\d+',b)))])
    return np.asarray(rows,dtype='float32')

In [6]:
_encoder=None;_encoder_tok=None
@torch.inference_mode()
def embed(texts,prefix='query: '):
    global _encoder,_encoder_tok
    if _encoder is None:
        path=locate('e5','intfloat/multilingual-e5-base')
        _encoder_tok=AutoTokenizer.from_pretrained(path)
        _encoder=AutoModel.from_pretrained(path).cuda().eval()
    texts=list(texts);output=[]
    for start in range(0,len(texts),32):
        batch=_encoder_tok([prefix+t for t in texts[start:start+32]],padding=True,truncation=True,max_length=256,return_tensors='pt').to('cuda')
        with torch.autocast('cuda',dtype=torch.float16):hidden=_encoder(**batch).last_hidden_state
        mask=batch.attention_mask.unsqueeze(-1)
        pooled=(hidden.float()*mask).sum(1)/mask.sum(1).clamp(min=1)
        output.append(torch.nn.functional.normalize(pooled,dim=1).cpu().numpy())
    return np.concatenate(output)

def release_encoder():
    global _encoder,_encoder_tok
    _encoder=None;_encoder_tok=None;gc.collect();torch.cuda.empty_cache()

In [7]:
def fine_tune_pairs(train,val,test,regression=False,label_names=('entailment','neutral','contradiction')):
    from datasets import Dataset,Value,disable_progress_bars
    from transformers import Trainer,TrainingArguments,DataCollatorWithPadding,set_seed
    from scipy.stats import spearmanr
    disable_progress_bars();set_seed(SEED);release_encoder()
    path=locate('indobert','indobenchmark/indobert-base-p1');tok=AutoTokenizer.from_pretrained(path)
    lengths=pd.Series([len(seq) for seq in tok(train.a.tolist(),train.b.tolist(),truncation=False)['input_ids']])
    show('Panjang token pasangan train',lengths.describe(percentiles=[.5,.9,.95,.99]))
    print('Train melewati PAIR_MAX_LEN (%):',lengths.gt(PAIR_MAX_LEN).mean()*100)
    def encode(frame,labels=True):
        payload={'a':frame.a.tolist(),'b':frame.b.tolist()}
        if labels:payload['labels']=frame.y.astype(float if regression else int).tolist()
        ds=Dataset.from_dict(payload)
        if labels and regression:ds=ds.cast_column('labels',Value('float32'))
        return ds.map(lambda v:tok(v['a'],v['b'],truncation=True,max_length=PAIR_MAX_LEN),batched=True,remove_columns=['a','b'])
    model=AutoModelForSequenceClassification.from_pretrained(path,num_labels=1 if regression else 3,
        problem_type='regression' if regression else 'single_label_classification',ignore_mismatched_sizes=True)
    if not regression:
        model.config.id2label=dict(enumerate(label_names));model.config.label2id={v:i for i,v in enumerate(label_names)}
    def metric(p):
        return {'selection':float(spearmanr(p.label_ids,p.predictions.ravel()).statistic)} if regression else {'selection':f1_score(p.label_ids,p.predictions.argmax(1),average='macro',zero_division=0)}
    args=TrainingArguments(output_dir=str(OUT/'checkpoints'),num_train_epochs=EPOCHS,learning_rate=2e-5,
        per_device_train_batch_size=16,per_device_eval_batch_size=32,fp16=True,eval_strategy='epoch',save_strategy='epoch',
        save_only_model=True,save_total_limit=1,load_best_model_at_end=True,metric_for_best_model='selection',greater_is_better=True,
        report_to='none',logging_strategy='epoch',disable_tqdm=True,seed=SEED)
    trainer=Trainer(model=model,args=args,train_dataset=encode(train),eval_dataset=encode(val),data_collator=DataCollatorWithPadding(tok),compute_metrics=metric)
    trainer.train();a=trainer.predict(encode(val,False)).predictions;b=trainer.predict(encode(test,False)).predictions
    trainer.save_model(str(OUT/'model_indobert'));tok.save_pretrained(str(OUT/'model_indobert'))
    del trainer,model;gc.collect();torch.cuda.empty_cache()
    return (a.ravel(),b.ravel()) if regression else (softmax(a,axis=1),softmax(b,axis=1))

In [8]:
@torch.inference_mode()
def pair_logits(repo,frame,folder=None,max_len=256):
    path=locate(folder,repo) if folder else repo
    tok=AutoTokenizer.from_pretrained(path);model=AutoModelForSequenceClassification.from_pretrained(path).cuda().eval()
    mapping={str(k).lower():int(v) for k,v in model.config.label2id.items()};output=[]
    for start in range(0,len(frame),32):
        batch=frame.iloc[start:start+32]
        enc=tok(batch.a.tolist(),batch.b.tolist(),padding=True,truncation=True,max_length=max_len,return_tensors='pt').to('cuda')
        with torch.autocast('cuda',dtype=torch.float16):logits=model(**enc).logits
        output.append(logits.float().cpu().numpy())
    del model;gc.collect();torch.cuda.empty_cache()
    return np.concatenate(output),mapping

## 02 · Data, grouping & overlap

Prioritas test resmi, lalu split komponen teks dari pool pengembangan. Semua teks identik di kedua sisi menjadi satu komponen; train tidak boleh berbagi endpoint dengan validation/test.

In [9]:
from datasets import load_dataset
data=load_dataset('LazarusNLP/stsb_mt_id')
pool=prepare_pairs(cap(data['validation'].to_pandas(),N_POOL),'text_1','text_2','correlation')
test=prepare_pairs(cap(data['test'].to_pandas(),N_TEST),'text_1','text_2','correlation')
assert pool.y.between(0,5).all() and test.y.between(0,5).all()
pool=purge(pool,endpoints(test));groups=pair_groups(pool)
a,b=next(GroupShuffleSplit(n_splits=1,test_size=.25,random_state=SEED).split(pool,groups=groups))
train,val=pool.iloc[a].copy(),pool.iloc[b].copy()
assert endpoints(train).isdisjoint(endpoints(val)) and endpoints(train).isdisjoint(endpoints(test)) and endpoints(val).isdisjoint(endpoints(test))
assert min(len(train),len(val),len(test))>=30
print('Train/validation/test:',len(train),len(val),len(test))

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]


Missing sebelum drop


,n,%
domain,0,0.0
data,0,0.0
type,0,0.0
score,0,0.0
correlation,0,0.0
text_1,0,0.0
text_2,0,0.0


Dibuang invalid: 0

Missing sebelum drop


,n,%
domain,0,0.0
data,0,0.0
type,0,0.0
score,0,0.0
correlation,0,0.0
text_1,0,0.0
text_2,0,0.0


Dibuang invalid: 0


Pasangan dikeluarkan karena teks lintas split: 120


/tmp/ipykernel_51/3948311101.py:22: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  return pd.factorize([root(x) for x in frame.a.map(key)])[0]


Train/validation/test: 1034 346 1379


## 03 · EDA wajib pasangan & skor

Train saja. Symmetry relevan untuk similarity, berbeda dengan NLI yang berarah.

In [10]:
text_eda(train.a,'Sisi A');text_eda(train.b,'Sisi B')
show('Distribusi skor',train.y.describe(percentiles=[.1,.25,.5,.75,.9]))
show('Distribusi bin skor',distribution(pd.cut(train.y,[-.01,1,2,3,4,5])))
show('Fitur pasangan',pd.DataFrame(pair_features(train),columns=['Jaccard','A_only','B_only','rasio_panjang','negasi_berbeda','angka_berbeda']).describe().T)
print('Pasangan identik:',train.a.map(key).eq(train.b.map(key)).sum(),'| duplicate pair:',train.duplicated(['a','b']).sum())
if 'data' in train:show('Sumber train',distribution(train['data']))


Sisi A kualitas


,0
n,1034
null,0
kosong/spasi,0
unik_terisi,1027
duplikat_ekstra_terisi,7



Sisi A length


,count,mean,std,min,1%,50%,90%,95%,99%,max
karakter,1034.0,74.0368,35.6514,14.0,26.0,65.0,122.0,141.0,188.34,228.0
kata,1034.0,10.6325,4.9343,3.0,4.0,10.0,17.0,20.0,26.00,30.0



Sisi A vocabulary sampel


,0
n_sample,1034.0000
token,10914.0000
unik,3325.0000
hapax,2043.0000
hapax_%_tipe,61.4436
hapax_%_token,18.7191


... (34 outputs left out)


Fitur pasangan


,count,mean,std,min,25%,50%,75%,max
Jaccard,1034.0,0.2729,0.2220,0.0000,0.0833,0.2308,0.4256,1.0
A_only,1034.0,0.6086,0.2678,0.0000,0.4000,0.6250,0.8421,1.0
B_only,1034.0,0.6094,0.2664,0.0000,0.4000,0.6250,0.8462,1.0
rasio_panjang,1034.0,1.0508,0.3452,0.3487,0.8174,1.0000,1.2166,3.0
negasi_berbeda,1034.0,0.1190,0.3239,0.0000,0.0000,0.0000,0.0000,1.0
angka_berbeda,1034.0,0.3433,1.2047,0.0000,0.0000,0.0000,0.0000,12.0


Pasangan identik: 8 | duplicate pair: 0

Sumber train


,n,%
data,,
answers-forums,272,26.3056
headlines,191,18.4720
MSRpar,187,18.0851
images,168,16.2476
MSRvid,129,12.4758
track5.en-en,87,8.4139


## 04 · Lima pendekatan

Semua parameter representasi/target dipelajari dari train. Model 1=mean, 2=TF-IDF cosine berkalibrasi linear train, 3=Ridge fitur simetris, 4=E5 cosine berkalibrasi train, 5=fine-tuned IndoBERT pair regressor.

In [11]:
from scipy.stats import pearsonr,spearmanr
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
def metrics(y,p):
    return {'pearson':float(pearsonr(y,p).statistic) if np.std(p)>1e-10 else np.nan,
        'spearman':float(spearmanr(y,p).statistic) if np.std(p)>1e-10 else np.nan,
        'MAE':float(np.abs(y-p).mean()),'RMSE':float(np.sqrt(np.mean((y-p)**2)))}
pred_val={};pred_test={}
pred_val['01_mean']=np.full(len(val),train.y.mean());pred_test['01_mean']=np.full(len(test),train.y.mean())
vectorizer=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=60000,sublinear_tf=True).fit(pd.concat([train.a,train.b]))
def cosine(frame):return np.asarray(vectorizer.transform(frame.a).multiply(vectorizer.transform(frame.b)).sum(1)).ravel()
fit=LinearRegression().fit(cosine(train)[:,None],train.y)
for frame,target in [(val,pred_val),(test,pred_test)]:target['02_tfidf']=fit.predict(cosine(frame)[:,None])
def symmetric(frame):
    reverse=frame.assign(a=frame.b,b=frame.a)
    return (pair_features(frame)+pair_features(reverse))/2
fit=Ridge(alpha=1).fit(symmetric(train),train.y)
pred_val['03_ridge']=fit.predict(symmetric(val));pred_test['03_ridge']=fit.predict(symmetric(test))

In [12]:
cosines=[]
for frame in [train,val,test]:cosines.append(np.sum(embed(frame.a)*embed(frame.b),axis=1))
fit=LinearRegression().fit(cosines[0][:,None],train.y)
pred_val['04_e5']=fit.predict(cosines[1][:,None]);pred_test['04_e5']=fit.predict(cosines[2][:,None])
# Pair augmentation hanya train. Prediction symmetry enforcement diuji terpisah.
aug=pd.concat([train,train.assign(a=train.b,b=train.a)],ignore_index=True)
both_test=pd.concat([test,test.assign(a=test.b,b=test.a)],ignore_index=True)
both_val=pd.concat([val,val.assign(a=val.b,b=val.a)],ignore_index=True)
vp,hp=fine_tune_pairs(aug,val,pd.concat([both_val,both_test],ignore_index=True),regression=True)
pred_val['05_indobert']=(hp[:len(val)]+hp[len(val):2*len(val)])/2
offset=2*len(val);pred_test['05_indobert']=(hp[offset:offset+len(test)]+hp[offset+len(test):])/2


Panjang token pasangan train


,0
count,2068.0000
mean,30.1518
std,12.8233
min,9.0000
50%,27.0000
90%,48.0000
95%,56.6500
99%,67.0000
max,83.0000


[transformers] You passed `num_labels=1` which is incompatible to the `id2label` map of length `5`.


Train melewati PAIR_MAX_LEN (%): 0.0


[transformers] BertForSequenceClassification LOAD REPORT from: /kaggle/input/notebooks/ivanwllm/gemastik-nlp-00-setup-offline/models/indobert
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


... (3 outputs left out)

{'eval_loss': '1.577', 'eval_selection': '0.7994', 'eval_runtime': '0.8719', 'eval_samples_per_second': '396.8', 'eval_steps_per_second': '6.881', 'epoch': '2'}


{'loss': '0.4515', 'grad_norm': '20.6', 'learning_rate': '1.026e-07', 'epoch': '3'}


{'eval_loss': '1.616', 'eval_selection': '0.7984', 'eval_runtime': '0.8927', 'eval_samples_per_second': '387.6', 'eval_steps_per_second': '6.721', 'epoch': '3'}


{'train_runtime': '58.14', 'train_samples_per_second': '106.7', 'train_steps_per_second': '3.354', 'train_loss': '1.161', 'epoch': '3'}


## 05 · Post-processing & pemilihan

Clipping 0–5; linear blend TF-IDF/E5/IndoBERT; isotonic calibration diuji cross-fit grup validation dan hanya dipakai jika RMSE membaik tanpa penurunan Spearman berarti. Holdout belum dinilai.

In [13]:
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import GroupKFold
for name in list(pred_val):
    pred_val[name]=np.clip(pred_val[name],0,5);pred_test[name]=np.clip(pred_test[name],0,5)
pred_val['06_blend']=sum(pred_val[k] for k in ['02_tfidf','04_e5','05_indobert'])/3
pred_test['06_blend']=sum(pred_test[k] for k in ['02_tfidf','04_e5','05_indobert'])/3
board=pd.DataFrame([{'model':name,**metrics(val.y.to_numpy(),p)} for name,p in pred_val.items()]).sort_values('spearman',ascending=False)
show('Validation',board);chosen=board.iloc[0]['model'];cross=np.zeros(len(val))
cal_groups=pair_groups(val);assert len(np.unique(cal_groups))>=3,'Validation perlu >=3 komponen teks untuk cross-fit; tambah data/revisi split.'
for a,b in GroupKFold(3).split(val,groups=cal_groups):
    cross[b]=IsotonicRegression(out_of_bounds='clip').fit(pred_val[chosen][a],val.y.iloc[a]).predict(pred_val[chosen][b])
before=metrics(val.y.to_numpy(),pred_val[chosen]);after=metrics(val.y.to_numpy(),cross)
use_iso=after['RMSE']<before['RMSE'] and after['spearman']>=before['spearman']-.01
cal=IsotonicRegression(out_of_bounds='clip').fit(pred_val[chosen],val.y)
final=cal.predict(pred_test[chosen]) if use_iso else pred_test[chosen]
cuts=np.linspace(0,5,101);target=val.y.to_numpy()>=PARAPHRASE_MIN
cut=max(cuts,key=lambda c:f1_score(target,pred_val[chosen]>=c,zero_division=0))
selection={'model':chosen,'isotonic':bool(use_iso),'paraphrase_cut':float(cut),'gold_paraphrase_min':PARAPHRASE_MIN}
(OUT/'selection.json').write_text(json.dumps(selection,indent=2));print(selection)


Validation


,model,pearson,spearman,MAE,RMSE
4,05_indobert,0.8127,0.8002,0.6624,0.8850
5,06_blend,0.7957,0.7861,0.7697,0.9339
2,03_ridge,0.6527,0.6482,0.9342,1.1282
1,02_tfidf,0.6280,0.6380,0.9573,1.1768
3,04_e5,0.6137,0.6338,0.9543,1.1804
0,01_mean,NaN,NaN,1.2756,1.4914


/tmp/ipykernel_51/3948311101.py:22: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  return pd.factorize([root(x) for x in frame.a.map(key)])[0]


{'model': '05_indobert', 'isotonic': False, 'paraphrase_cut': 4.0, 'gold_paraphrase_min': 4.0}


## 06 · Holdout & kelemahan

Sumber, overlap tinggi tetapi skor rendah, angka/negasi berbeda, error absolut dan bias rentang skor. Threshold parafrase dipilih dari validation pada skor RAW model pilihan, bukan pada skala isotonic.

In [14]:
table=pd.DataFrame([{'model':name,**metrics(test.y.to_numpy(),p)} for name,p in pred_test.items()]+[{'model':'final_selected',**metrics(test.y.to_numpy(),final)}])
show('Holdout',table);table.to_csv(OUT/'models.csv',index=False)
audit=test.assign(pred=final,error=np.abs(test.y-final),paraphrase_pred=pred_test[chosen]>=cut)
print('F1 parafrase:',f1_score(test.y>=PARAPHRASE_MIN,audit.paraphrase_pred,zero_division=0))
audit['overlap']=pair_features(test)[:,0];audit['bin_score']=pd.cut(audit.y,[-.01,1,2,3,4,5])
show('Bias menurut skor',audit.groupby('bin_score',observed=True)[['y','pred','error']].agg(['size','mean']))
if 'data' in audit:
    rows=[{'sumber':name,'n':len(g),**metrics(g.y.to_numpy(),g.pred.to_numpy())} for name,g in audit.groupby('data') if len(g)>=3]
    show('Per sumber',pd.DataFrame(rows))
show('High overlap, low gold',audit[(audit.overlap>.6)&(audit.y<2)][['a','b','y','pred']].head(TOP))
show('Error terbesar',audit.nlargest(TOP,'error')[['a','b','y','pred','error']])
audit.to_csv(OUT/'predictions.csv',index=False)


Holdout


,model,pearson,spearman,MAE,RMSE
0,01_mean,NaN,NaN,1.3290,1.5367
1,02_tfidf,0.5762,0.5665,1.0111,1.2473
2,03_ridge,0.5658,0.5546,1.0338,1.2723
3,04_e5,0.6459,0.6541,0.9658,1.1745
4,05_indobert,0.7803,0.7527,0.7378,0.9666
5,06_blend,0.7773,0.7627,0.8158,1.0006
6,final_selected,0.7803,0.7527,0.7378,0.9666


F1 parafrase: 0.48739495798319327

Bias menurut skor


y         pred         error        
             size    mean size    mean  size    mean
bin_score                                           
(-0.01, 1.0]  308  0.4247  308  1.0213   308  0.6997
(1.0, 2.0]    226  1.7000  226  2.2110   226  0.8342
(2.0, 3.0]    259  2.6778  259  3.0110   259  0.7965
(3.0, 4.0]    355  3.6675  355  3.4564   355  0.6081
(4.0, 5.0]    231  4.7005  231  3.9133   231  0.8278


Per sumber


,sumber,n,pearson,spearman,MAE,RMSE
0,MSRpar,250,0.6321,0.5941,0.5767,0.7290
1,MSRvid,250,0.8863,0.8824,0.6246,0.8523
2,answer-answer,254,0.6133,0.6151,1.1808,1.3980
3,headlines,250,0.7871,0.7817,0.7295,0.9558
4,images,250,0.8713,0.8328,0.5909,0.7750
5,track5.en-en,125,0.7887,0.7938,0.6965,0.8901



High overlap, low gold


,a,b,y,pred
312,Piala Dunia langsung: Prancis 0-0 Jerman,Piala Dunia langsung: Jerman 0-0 Ghana,1.400,3.9682
320,Seorang pria sedang memainkan gitar.,Seorang pria sedang memainkan terompet.,1.714,2.2627
496,Seorang wanita sedang memotong bawang.,Seorang wanita sedang memotong tahu.,1.800,3.3815
696,Seorang pria sedang memainkan gitar.,Seorang pria sedang memainkan terompet.,1.714,2.2627
948,Seorang pria sedang memainkan gitar.,Seorang pria sedang memainkan keyboard.,1.800,4.0760
1041,Seorang pria sedang memainkan harpa.,Seorang pria sedang memainkan keyboard.,1.500,2.5421



Error terbesar


,a,b,y,pred,error
962,Anda melakukannya dengan benar.,"Ya, Anda mengerti.",5.00,1.0998,3.9002
581,Apakah kamu bisa melakukan ini?,Apakah Anda bisa?,5.00,1.6751,3.3249
1220,"Ya, Anda harus mengajukan pengembalian pajak di Kanada.",Anda tidak diharuskan mengajukan pengembalian pajak di Kanada jika Anda tidak memiliki penghasilan kena pajak.,1.00,4.2827,3.2827
551,Terserah Anda.,Ini terserah kamu.,5.00,1.8155,3.1845
290,Anda melakukannya dengan benar.,Anda benar.,5.00,1.8242,3.1758
1263,"Ya, ada aturan yang melarang ini.",Tidak ada aturan yang melarangnya.,1.00,4.0285,3.0285
449,Anda berada di jalan yang benar.,Anda tepat sasaran.,4.00,1.0572,2.9428
544,Hawaii meloloskan RUU pernikahan gay,Senat AS meloloskan RUU pekerja gay,1.00,3.9087,2.9087
213,Anda juga bisa melakukannya.,Ya kamu bisa melakukannya.,5.00,2.0968,2.9032
484,Kamu tidak perlu khawatir.,Anda tidak perlu khawatir.,5.00,2.2151,2.7849


## Interpretasi & sumber
Semua angka dihitung saat run; pemilihan validation tidak diubah setelah holdout. E5 memakai query: di kedua sisi STS, query:/passage: pada retrieval. [E5](https://huggingface.co/intfloat/multilingual-e5-base), [IndoBERT](https://huggingface.co/indobenchmark/indobert-base-p1), [IndoNLI](https://github.com/ir-nlp-csui/indonli).

In [15]:
import importlib.metadata as metadata
manifest={'task':RESULT_NAME,'seconds':time.perf_counter()-START,'gpu':torch.cuda.get_device_name(0),'seed':SEED,'versions':{p:metadata.version(p) for p in ['torch','transformers','scikit-learn','pandas']}}
(OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
print(manifest)

{'task': 'hasil_similarity_review', 'seconds': 136.38809173799996, 'gpu': 'Tesla T4', 'seed': 42, 'versions': {'torch': '2.11.0+cu130', 'transformers': '5.20.2', 'scikit-learn': '1.6.1', 'pandas': '2.3.3'}}
